## Worker 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.04.02 </div>
<div style="text-align: right"> last update : 2026.04.02 </div>

개정 이력  
- `2026.04.02` : 노트북 최초 작성 — ReAct Worker 아키텍처 반영 (구 03_node_test.ipynb 대체)

### 개요

현재 에이전트의 각 분석 단계는 **ReAct Worker**로 구현되어 있습니다.  
Worker는 구버전의 단순 노드 함수(`make_*_node()`)와 달리, **자체 Tool loop를 가진 ReAct sub-agent**입니다.

| Worker | Factory | Tools |
|--------|---------|-------|
| deal_structuring | `make_deal_structuring_worker_node()` | `search_company_context`, `fetch_notion_deal`, `format_currency` |
| scoring | `make_scoring_worker_node()` | `get_scoring_criteria`, `search_company_context`, `calculate_weighted_score`, `search_similar_projects` |
| resource_estimation | `make_resource_estimation_worker_node()` | `get_team_members`, `get_company_settings`, `search_similar_projects`, `estimate_timeline`, `calculate_roi` |
| risk_analysis | `make_risk_analysis_worker_node()` | `search_company_context`, `web_search`, `assess_risk_matrix`, `search_similar_projects` |
| similar_project | `make_similar_project_worker_node()` | `search_similar_projects`, `get_past_deal_analysis`, `search_company_context` |
| final_verdict | `make_final_verdict_worker_node()` | `calculate_weighted_score`, `calculate_roi` |

**핵심 변경점 (vs 구버전 노드):**
- 구: `make_*_node(context_store, project_store)` → 직접 LLM 호출
- 신: `make_*_worker_node(parent_log_id)` → 내부에서 ReAct agent가 Tool을 자율적으로 호출

**사전 요구사항:** `make docker-up && make migrate && make seed` + Pinecone 설정 + `.env` API keys

In [ ]:
import json
import sys
import uuid

from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, "../../")

### 0. 인프라 초기화

In [ ]:
# ToolContext 초기화 — 모든 Worker의 Tool이 이에 의존
from backend.app.agent.tools._context import init_tool_context
from backend.app.db.session import AsyncSessionLocal
from backend.app.db.vector_store import CompanyContextStore, ProjectHistoryStore

init_tool_context(
    session_factory=AsyncSessionLocal,
    company_context_store=CompanyContextStore(),
    project_history_store=ProjectHistoryStore(),
)
print("✓ ToolContext 초기화 완료")

In [ ]:
# 테스트용 deal_id 및 샘플 딜 입력
test_deal_id = str(uuid.uuid4())

sample_deal_input = """[딜 기본 정보]
고객사: 메가테크
고객 규모: 대기업
산업군: 제조업
프로젝트명: 공장 설비 예지보전 AI 시스템 개발
예상 금액: 20000만원
금액 단위: 만원
기간: 5개월

[상세 내용]
## 프로젝트 배경
기존 정기 점검 체계에서 AI 기반 예측 유지보수로 전환하여 설비 가동률 향상 및 유지보수 비용 절감을 목표로 함.
현재 설비 고장으로 인한 비계획 정지가 연간 약 50회 발생하며, 이를 80% 이상 감소시키는 것이 핵심 목표.

## 기술 요구사항
- Python, TensorFlow 기반 시계열 예측 모델
- IoT 센서 데이터 실시간 수집/처리 파이프라인
- 대시보드 (React 기반)
- MLOps: 모델 재학습 자동화 파이프라인

## 데이터 요구사항
- 3년간 설비 센서 데이터 (진동, 온도, 전류) 약 500GB
- 설비 고장 이력 데이터

## 보안 요구사항
- 공장 내 보안 네트워크 환경에서만 운영
- 온프레미스 배포 필수
- 외부 클라우드 연결 불가

## 결제 조건
착수금 30%, 중간 40%, 완료 30%

## 이해관계자
- 발주측 PM: 설비관리팀 김부장
- 의사결정권자: 제조본부장
- IT 지원: 자체 IT 인프라팀 보유"""

print(f"deal_id: {test_deal_id}")
print(f"deal_input 길이: {len(sample_deal_input)} chars")

---
### 1. deal_structuring Worker

원본 딜 텍스트에서 구조화된 JSON을 추출합니다.  
Worker는 `search_company_context`, `fetch_notion_deal`, `format_currency` Tool을 자율적으로 사용합니다.

In [ ]:
from backend.app.agent.workers.deal_structuring import make_deal_structuring_worker_node

deal_structuring_node = make_deal_structuring_worker_node(parent_log_id=None)

# AgentState 구성
state = {
    "deal_id": test_deal_id,
    "deal_input": sample_deal_input,
}

result = await deal_structuring_node(state)
print(f"status: {result.get('status')}")
print(f"errors: {result.get('errors', [])}")

In [ ]:
structured_deal = result.get("structured_deal", {})
print("=== Structured Deal ===")
print(json.dumps(structured_deal, ensure_ascii=False, indent=2))

# 핵심 필드 검증
print(f"\n=== 핵심 필드 검증 ===")
print(f"customer_name: {structured_deal.get('customer_name')}")
print(f"customer_industry: {structured_deal.get('customer_industry')}")
print(f"expected_amount: {structured_deal.get('expected_amount')}만원 (unit: {structured_deal.get('amount_unit')})")
print(f"duration_months: {structured_deal.get('duration_months')}")
print(f"missing_fields: {structured_deal.get('missing_fields', [])}")

# amount_unit은 normalize_amount_to_manwon에 의해 '만원'이어야 함
assert structured_deal.get('amount_unit') == '만원', f"amount_unit이 '만원'이어야 함, got: {structured_deal.get('amount_unit')}"

---
### 2. Hold 조건 체크

`MISSING_FIELDS_THRESHOLD` (3개) 이상의 critical field가 누락되면 hold_recommended가 됩니다.  
6개 critical fields: `customer_name`, `customer_industry`, `project_overview`, `tech_requirements`, `expected_amount`, `duration_months`

In [ ]:
from backend.app.agent.base import CRITICAL_FIELDS, MISSING_FIELDS_THRESHOLD

# 정보 부족 딜로 hold 조건 테스트
hold_deal_input = """[딜 기본 정보]
프로젝트명: AI 프로젝트

[상세 내용]
AI를 활용한 프로젝트를 진행하고 싶습니다. 상세 내용은 추후 공유 예정입니다."""

hold_state = {
    "deal_id": str(uuid.uuid4()),
    "deal_input": hold_deal_input,
}

hold_node = make_deal_structuring_worker_node(parent_log_id=None)
hold_result = await hold_node(hold_state)

hold_deal = hold_result.get("structured_deal", {})
missing = hold_deal.get("missing_fields", [])
hold_recommended = len(missing) >= MISSING_FIELDS_THRESHOLD

print(f"missing_fields ({len(missing)}개): {missing}")
print(f"MISSING_FIELDS_THRESHOLD: {MISSING_FIELDS_THRESHOLD}")
print(f"CRITICAL_FIELDS: {CRITICAL_FIELDS}")
print(f"Hold 권고: {hold_recommended}")

---
### 3. resource_estimation Worker

팀 구성, 일정, 비용, 수익성을 추정합니다.  
Worker는 `get_team_members`, `get_company_settings`, `search_similar_projects`, `estimate_timeline`, `calculate_roi` Tool을 사용합니다.

In [ ]:
from backend.app.agent.workers.resource_estimation import make_resource_estimation_worker_node

resource_node = make_resource_estimation_worker_node(parent_log_id=None)

# deal_structuring 결과를 state에 누적
state_with_deal = {
    "deal_id": test_deal_id,
    "deal_input": sample_deal_input,
    "structured_deal": structured_deal,
}

resource_result = await resource_node(state_with_deal)
print(f"errors: {resource_result.get('errors', [])}")

In [ ]:
resource_estimate = resource_result.get("resource_estimate", {})
print("=== Resource Estimation ===")
print(json.dumps(resource_estimate, ensure_ascii=False, indent=2))

# 핵심 필드 검증
print(f"\n=== 핵심 검증 ===")
if "profitability" in resource_estimate:
    prof = resource_estimate["profitability"]
    print(f"deal_amount: {prof.get('deal_amount')}만원")
    print(f"estimated_cost: {prof.get('estimated_cost')}만원")
    print(f"expected_margin: {prof.get('expected_margin')}")
if "duration_months" in resource_estimate:
    print(f"duration_months: {resource_estimate['duration_months']}")
    print(f"duration_with_buffer: {resource_estimate.get('duration_with_buffer')}")

---
### 4. similar_project Worker

과거 유사 프로젝트를 검색하고 비교 분석합니다.  
Worker는 `search_similar_projects`, `get_past_deal_analysis`, `search_company_context` Tool을 사용합니다.

In [ ]:
from backend.app.agent.workers.similar_project import make_similar_project_worker_node

similar_node = make_similar_project_worker_node(parent_log_id=None)

similar_result = await similar_node(state_with_deal)
print(f"errors: {similar_result.get('errors', [])}")

In [ ]:
similar_projects = similar_result.get("similar_projects", [])
print(f"유사 프로젝트 수: {len(similar_projects)}")
for p in similar_projects:
    print(f"  {p.get('project_name', 'N/A')}")
    print(f"    relevance_score: {p.get('relevance_score', 'N/A')}")
    print(f"    lessons_learned: {str(p.get('lessons_learned', ''))[:100]}...")

---
### 5. scoring Worker

7개 기준으로 딜을 평가합니다.  
Worker는 `get_scoring_criteria`, `calculate_weighted_score` 등의 Tool을 사용합니다.  
최종적으로 `recalculate_scores()`와 `determine_verdict()`로 서버사이드 재계산을 수행합니다.

In [ ]:
from backend.app.agent.workers.scoring import make_scoring_worker_node

scoring_node = make_scoring_worker_node(parent_log_id=None)

# resource_estimate를 state에 추가 (scoring이 참조)
state_with_resource = {
    **state_with_deal,
    "resource_estimate": resource_estimate,
}

scoring_result = await scoring_node(state_with_resource)
print(f"errors: {scoring_result.get('errors', [])}")

In [ ]:
scores = scoring_result.get("scores", [])
total_score = scoring_result.get("total_score", 0.0)
verdict = scoring_result.get("verdict", "pending")

print(f"=== Scoring 결과 ===")
print(f"총점: {total_score}")
print(f"판정: {verdict}")
print(f"\n개별 점수:")
for s in scores:
    print(f"  {s['criterion']}: {s['score']}점 × {s['weight']} = {s['weighted_score']}")

---
### 6. risk_analysis Worker

5개 카테고리(기술/일정/재무/고객/범위)의 리스크를 식별합니다.  
Worker는 `search_company_context`, `web_search`, `assess_risk_matrix`, `search_similar_projects` Tool을 사용합니다.

In [ ]:
from backend.app.agent.workers.risk_analysis import make_risk_analysis_worker_node

risk_node = make_risk_analysis_worker_node(parent_log_id=None)

risk_result = await risk_node(state_with_resource)
print(f"errors: {risk_result.get('errors', [])}")

In [ ]:
risks = risk_result.get("risks", [])
interdeps = risk_result.get("risk_interdependencies", [])

print(f"=== 리스크 분석 결과 ===")
print(f"리스크 수: {len(risks)}")
for r in risks:
    print(f"  [{r.get('level', 'N/A')}] {r.get('category', '')} — {r.get('item', '')}")

print(f"\n리스크 상호작용 ({len(interdeps)}건):")
for ri in interdeps:
    print(f"  {ri.get('risk_pair', [])} → {ri.get('combined_effect', '')[:80]}")

---
### 7. final_verdict Worker

모든 분석 결과를 종합하여 8개 섹션 마크다운 보고서를 생성합니다.  
Worker는 `calculate_weighted_score`, `calculate_roi` Tool로 수치를 재검증합니다.

In [ ]:
from backend.app.agent.workers.final_verdict import make_final_verdict_worker_node

verdict_node = make_final_verdict_worker_node(parent_log_id=None)

# 모든 이전 Worker 결과를 state에 누적
full_state = {
    "deal_id": test_deal_id,
    "deal_input": sample_deal_input,
    "structured_deal": structured_deal,
    "scores": scores,
    "total_score": total_score,
    "verdict": verdict,
    "resource_estimate": resource_estimate,
    "risks": risks,
    "risk_interdependencies": interdeps,
    "similar_projects": similar_projects,
}

verdict_result = await verdict_node(full_state)
print(f"errors: {verdict_result.get('errors', [])}")

In [ ]:
final_report = verdict_result.get("final_report", "")
print(f"리포트 길이: {len(final_report)} chars")

# 8개 섹션 존재 확인
expected_sections = ["핵심 결론", "Deal 개요", "Deal 선정 기준 적합성", "평가 상세",
                     "예상 작업 범위", "리스크 분석", "유사 프로젝트", "권고사항"]
for section in expected_sections:
    status = "✓" if section in final_report else "✗"
    print(f"  {status} '{section}' 섹션")

In [ ]:
# 마크다운 렌더링
from IPython.display import Markdown, display

display(Markdown(final_report))

---
### 8. 전체 State 요약

모든 Worker 실행 후 누적된 state를 확인합니다.

In [ ]:
# 전체 파이프라인 결과 요약
print("=== 파이프라인 결과 요약 ===")
print(f"structured_deal: {len(json.dumps(structured_deal))} chars")
print(f"scores: {len(scores)}개 기준")
print(f"total_score: {total_score}")
print(f"verdict: {verdict}")
print(f"resource_estimate: {len(json.dumps(resource_estimate))} chars")
print(f"risks: {len(risks)}개")
print(f"risk_interdependencies: {len(interdeps)}개")
print(f"similar_projects: {len(similar_projects)}개")
print(f"final_report: {len(final_report)} chars")